# Simulating Roman star images (`psf_fomo.sim`)

This notebook walks through the simulation module step by step: how to make a simulation,
which settings it takes, and what can be added to it (PSF errors, their variations, noise,
WCS, colour). Each step shows the result.

Every simulation follows the same pattern:

1. **create** a `RomanSimulation` with the common settings,
2. **configure** it with `set_*` / `add_*` methods,
3. **run** it, which returns a `Simulation` (images, catalogs, truth).

Units and definitions (pixels, moments, the "oversampled image") are in
[`docs/conventions.md`](../docs/conventions.md); all options are in
[`docs/sim.md`](../docs/sim.md). Nothing in this notebook is written to disk; the cells that
save are commented out.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import galsim
import galsim.roman
import yaml
from astropy.table import Table
from IPython.display import display

from psf_fomo.sim import RomanSimulation
from psf_fomo.moments.measure import make_star, measure_star
from psf_fomo.io import to_plain

plt.rcParams["figure.dpi"] = 80

A few plotting helpers used below. `truth` draws the noiseless truth PSF of one star on a grid
finer than the native pixels; `hsm` measures its size `T` and shape `e1`, `e2` (see the
conventions page).

In [ ]:
def truth(sim, star_id=0, obs=0, oversample=5, include_pixel=True):
    # Unit-flux truth PSF of one star, `oversample` × finer than the native pixels.
    cat = sim.catalogs[obs]
    sca = int(cat[cat["star_id"] == star_id]["sca"][0])
    return sim.truth_psf(sca, obs, star_id, oversample=oversample, include_pixel=include_pixel)


def hsm(image):
    m = measure_star(make_star(image, image.true_center), "hsm")
    return {"T [pix²]": m["T"], "e1": m["e1"], "e2": m["e2"]}


def show(images, titles, log=True, suptitle=None):
    # Images side by side; log scale = log10(image / peak).
    fig, axes = plt.subplots(1, len(images), figsize=(3 * len(images), 3), squeeze=False)
    for ax, im, title in zip(axes[0], images, titles):
        a = im.array if hasattr(im, "array") else np.asarray(im)
        ax.imshow(np.log10(np.maximum(a / a.max(), 1e-5)) if log else a, origin="lower")
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle)
    fig.tight_layout()
    plt.show()


def show_differences(reference, images, titles, suptitle=None):
    # (image − reference) / peak of the reference, symmetric colour scale per panel.
    fig, axes = plt.subplots(1, len(images), figsize=(3 * len(images), 3), squeeze=False)
    for ax, im, title in zip(axes[0], images, titles):
        d = (im.array - reference.array) / reference.array.max()
        lim = np.abs(d).max() or 1
        ax.imshow(d, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle)
    fig.tight_layout()
    plt.show()

## 1. The smallest simulation

With no settings: SCA 1, one observation, one star at a random position, magnitude 18 (AB),
filter H158, the Roman optics with no extra aberrations drawn chromatically (integrated over
the bandpass with a flat SED), the charge-diffusion (MTF) kernel, Gaussian noise and a simple
0.11″ pixel-scale WCS.

In [ ]:
sim = RomanSimulation(seed=0).run()
print("images:", list(sim.images), "(keys are (observation, SCA))")
print("image shape:", sim.images[0, 1].array.shape, "pixels, in ADU")
sim.catalogs[0]

The catalog has one row per star: position (pixel centres at integers, the SCA spans
1 … 4088), sub-pixel phase, sky position, magnitude, flux, and the PSF errors and extra
aberrations used for that star.

`plot_sca` shows the full SCA image with the stars marked. `plot_truth_vs_image` re-draws the
star without noise from the configuration and seed and subtracts it from the image: every
star can be regenerated exactly.

In [ ]:
sim.plot_sca()
sim.plot_truth_vs_image(star_id=0);

## 2. The main arguments

`RomanSimulation(scas, n_obs, n_stars, filter, seed, stamp_size)`:

- `scas`: which detectors (any of 1 … 18), one image each;
- `n_obs`: number of observations (repeated exposures; see section 8);
- `n_stars`: stars per SCA;
- `filter`: a Roman filter (`R062`, `Z087`, `Y106`, `J129`, `H158`, `F184`, `K213`, `W146`);
- `seed`: every random draw (positions, magnitudes, noise, errors) follows from it;
- `stamp_size`: the size in pixels of the star cut-outs used by the other modules.

In [ ]:
sim = RomanSimulation(n_stars=40, seed=1, stamp_size=25).run()
sim.plot_sca()
sim.plot_stamps(n_max=8);

The filter sets the wavelength, and so the size of the PSF relative to the fixed 0.11″ pixel.
Below: the same star in four filters, as the oversampled truth PSF (top) and as the native
pixels of the image (bottom).

In [ ]:
filters = ["Y106", "J129", "H158", "F184"]
oversampled, native = [], []
for f in filters:
    s = RomanSimulation(seed=2, filter=f).run()
    oversampled.append(truth(s))
    native.append(s.render_stamp(0, size=11))
show(oversampled, filters, suptitle="oversampled truth PSF, log10(image / peak)")
show(native, filters, log=False, suptitle="native pixels, noiseless (linear)")

## 3. Where the stars go: placement and sub-pixel phase

- `set_placement("random")` (default): uniform positions, so random sub-pixel phases.
- `set_placement("grid")`: an evenly spaced grid, each star with a random sub-pixel phase.
- `phase=(fx, fy)`: every star at the same sub-pixel phase (in pixels, −0.5 … 0.5).
- `edge_margin`: minimum distance in pixels from the SCA edge.

Top: positions on the SCA. Bottom: sub-pixel phases.

In [ ]:
placements = {"random": dict(kind="random"),
              "grid": dict(kind="grid"),
              "grid, phase (0, 0)": dict(kind="grid", phase=(0.0, 0.0)),
              "random, edge_margin 800": dict(kind="random", edge_margin=800)}
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
for k, (name, kwargs) in enumerate(placements.items()):
    s = RomanSimulation(n_stars=49, seed=1)
    s.set_placement(**kwargs)
    cat = s.run().catalogs[0]
    axes[0, k].scatter(cat["x"], cat["y"], s=8)
    axes[0, k].set(xlim=(0, 4089), ylim=(0, 4089), aspect="equal", title=name,
                   xlabel="x [pix]", ylabel="y [pix]")
    axes[1, k].scatter(cat["phase_x"], cat["phase_y"], s=8)
    axes[1, k].set(xlim=(-0.5, 0.5), ylim=(-0.5, 0.5), aspect="equal",
                   xlabel="phase x [pix]", ylabel="phase y [pix]")
fig.tight_layout()
plt.show()

The sub-pixel phase changes how the same PSF falls on the native pixels. Below: one star at
four fixed phases, noiseless, native pixels.

In [ ]:
phases = [(0.0, 0.0), (0.25, 0.0), (0.5, 0.0), (0.5, 0.5)]
stamps = []
for phase in phases:
    s = RomanSimulation(seed=2)
    s.set_placement("grid", phase=phase)
    stamps.append(s.run().render_stamp(0, size=11))
show(stamps, [f"phase {p}" for p in phases], log=False)

## 4. Brightness and noise

**Brightness** is set in AB magnitudes: `set_brightness(mag=18)` for all stars, or
`set_brightness(mag_range=(17, 21))` for magnitudes drawn uniformly in a range. They are
converted to electrons with the Roman zeropoint, collecting area and exposure time.

**Noise**:
- `set_noise("gaussian", sigma=...)`: white Gaussian noise, σ in e⁻ per pixel (default).
- `set_noise("roman", effects=[...])`: the noise and detector effects of `galsim.roman`, each
  switchable: `sky`, `poisson`, `reciprocity`, `dark`, `nonlinearity`, `ipc`, `read`,
  `quantize` (default: all). The expected sky and dark levels are subtracted afterwards.

Images are in ADU (e⁻ / `galsim.roman.gain`). The plots below show magnitude against flux and
against the S/N of each stamp.

In [ ]:
s = RomanSimulation(n_stars=60, seed=3)
s.set_brightness(mag_range=(16, 23))
s.set_noise("gaussian", sigma=10.0)
bright = s.run()
cat = bright.catalogs[0]
cat["snr"] = [np.sqrt(np.sum(bright.render_stamp(int(r["star_id"])).array ** 2) / r["bkg_var"])
              for r in cat]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].semilogy(cat["mag"], cat["flux"], "o", ms=3)
axes[0].set(xlabel="AB magnitude", ylabel="flux [e⁻]")
axes[1].semilogy(cat["mag"], cat["snr"], "o", ms=3)
axes[1].set(xlabel="AB magnitude", ylabel="stamp S/N (σ = 10 e⁻)")
fig.tight_layout()
plt.show()

The same star (magnitude 19) with different noise settings: the stamp, and a histogram of
image − noiseless image in a region without stars.

In [ ]:
noise_cases = {"gaussian σ = 1 e⁻": dict(kind="gaussian", sigma=1.0),
               "gaussian σ = 20 e⁻": dict(kind="gaussian", sigma=20.0),
               "roman: read only": dict(kind="roman", effects=["read"]),
               "roman: all effects": dict(kind="roman")}
fig, axes = plt.subplots(2, len(noise_cases), figsize=(3.3 * len(noise_cases), 6.3))
for k, (name, kwargs) in enumerate(noise_cases.items()):
    s = RomanSimulation(seed=15)
    s.set_brightness(mag=19)
    s.set_noise(**kwargs)
    r = s.run(save_noiseless=True)
    bounds = r.render_stamp(0).bounds
    axes[0, k].imshow(r.images[0, 1][bounds].array, origin="lower")
    axes[0, k].set_title(name, fontsize=9)
    axes[0, k].axis("off")
    box = galsim.BoundsI(100, 400, 100, 400)          # away from the star
    if (bounds & box).isDefined():
        box = galsim.BoundsI(3600, 3900, 3600, 3900)
    noise = (r.images[0, 1][box].array - r.noiseless[0, 1][box].array).ravel()
    axes[1, k].hist(noise, bins=60, histtype="step")
    axes[1, k].set_xlabel("image − noiseless [ADU]")
    del r
fig.tight_layout()
plt.show()

## 5. The truth PSF

Each star's PSF can be drawn without noise in two ways:

- `render_stamp(star_id)`: the star as it is in the image (native pixels, flux included);
- `truth_psf(sca, obs, star_id, oversample, include_pixel)`: unit flux on a grid `oversample`
  times finer. With `include_pixel=True` (default) this is the **oversampled image** used
  throughout the package: the PSF convolved with the native pixel, sampled more finely. With
  `include_pixel=False` it is the PSF before the pixel.

The printed table gives the HSM size and shape of each image.

In [ ]:
r = RomanSimulation(seed=5).run()
images = {"native stamp": r.render_stamp(0, size=25),
          "oversampled ×5, pixel included": truth(r),
          "oversampled ×5, before the pixel": truth(r, include_pixel=False)}
show(list(images.values()), list(images), suptitle="log10(image / peak)")
Table(rows=[{"image": name, **hsm(im)} for name, im in list(images.items())[1:]])

## 6. PSF errors, one at a time

Errors change the truth PSF; they are what the PSF modelling has to recover. The truth is
built in this order:

optical PSF (Roman optics + extra aberrations) → dilation → shear → Gaussian kernel → jitter
→ MTF → pixel → noise.

| method | what it does |
|---|---|
| `add_dilation(value)` | scales the optical PSF (and the MTF) by `value` |
| `add_shear(g1, g2)` | shears the optical PSF (reduced shear) |
| `set_mtf(scale)` / `set_mtf(None)` | width of the charge-diffusion kernel, or no MTF |
| `add_kernel(mxx, mxy, myy)` | convolves with a Gaussian of second moments M (native pix²) |
| `set_zernikes({noll: waves})` | extra optical aberrations |
| `add_jitter(...)` | pointing jitter during the exposure |
| `add_aberration_scatter(sigma)` | independent extra aberrations for each star |

Each case below is the same star (same seed) with one error. The plots show
(PSF with the error − PSF without) / peak, on the oversampled grid; the table gives the HSM
size and shape.

In [ ]:
def one_star_truth(configure=None, **kwargs):
    # Oversampled truth PSF of one star with one setting changed (the run itself is dropped).
    s = RomanSimulation(seed=10, **kwargs)
    if configure is not None:
        configure(s)
    return truth(s.run())


nominal = one_star_truth()
error_cases = {
    "dilation 1.02": lambda s: s.add_dilation(1.02),
    "shear g1 = 0.05": lambda s: s.add_shear(g1=0.05),
    "MTF scale 1.2": lambda s: s.set_mtf(1.2),
    "kernel M = (0.03, 0.02, 0.03)": lambda s: s.add_kernel(0.03, 0.02, 0.03),
}
error_psfs = {name: one_star_truth(f) for name, f in error_cases.items()}
show_differences(nominal, list(error_psfs.values()), list(error_psfs))
Table(rows=[{"error": "none", **hsm(nominal)}]
      + [{"error": name, **hsm(im)} for name, im in error_psfs.items()])

### Extra aberrations

`set_zernikes({noll: waves})` adds aberrations to Roman's design, by Noll index (4 defocus,
5–6 astigmatism, 7–8 coma, 11 spherical, up to `max_zernike`). Top: the oversampled truth
PSF (log scale); bottom: the difference to the nominal PSF.

In [ ]:
zernike_cases = {"Z4 = 0.05": {4: 0.05}, "Z5 = 0.05": {5: 0.05}, "Z7 = 0.05": {7: 0.05}}
zernike_psfs = {n: one_star_truth(lambda s, v=v: s.set_zernikes(v))
                for n, v in zernike_cases.items()}
show(list(zernike_psfs.values()), list(zernike_psfs), suptitle="log10(image / peak)")
show_differences(nominal, list(zernike_psfs.values()), list(zernike_psfs))

### Jitter

The PSF is averaged over the pointing offsets during the exposure:

- `add_jitter("random_walk", rms_mas=..., n_steps=...)`: a new random walk in each
  observation, scaled to the given RMS per axis. The default `rms_mas` is a **placeholder**
  (GalSim's `galsim.roman.jitter_rms`), not a measured Roman value.
- `add_jitter("timeseries", t=..., dx_mas=..., dy_mas=...)`: your own pointing trajectory
  (seconds, milliarcsec along the world axes), the same in every observation.

Top: the distribution of pointing offsets for each case. Bottom: the difference to the PSF
without jitter.

In [ ]:
t = np.linspace(0, galsim.roman.exptime, 200)
jitter_cases = {
    "random walk, default rms": lambda s: s.add_jitter("random_walk"),
    "random walk, 40 mas": lambda s: s.add_jitter("random_walk", rms_mas=40.0),
    "drift along u": lambda s: s.add_jitter("timeseries", t=t, dx_mas=np.linspace(-60, 60, t.size),
                                           dy_mas=np.zeros(t.size)),
}
fig, axes = plt.subplots(1, len(jitter_cases), figsize=(3.3 * len(jitter_cases), 3.3))
jitter_psfs = {}
for ax, (name, configure) in zip(axes, jitter_cases.items()):
    s = RomanSimulation(seed=10)
    configure(s)
    r = s.run()
    kernel = r.truth.obs_truth(0)["jitter"]          # shifted delta functions, arcsec
    density = galsim.Convolve(kernel, galsim.Gaussian(sigma=0.001)).drawImage(
        nx=201, ny=201, scale=0.001, method="no_pixel")
    ax.imshow(density.array, origin="lower", extent=(-100.5, 100.5, -100.5, 100.5))
    ax.set(title=name, xlabel="Δu [mas]", ylabel="Δv [mas]")
    jitter_psfs[name] = truth(r)
    del r
fig.tight_layout()
plt.show()
show_differences(nominal, list(jitter_psfs.values()), list(jitter_psfs))

### Aberrations that differ from star to star

`add_aberration_scatter(sigma)` adds an independent Gaussian draw (σ in waves) to every
extra Zernike of every star. Each star then needs its own optical PSF, which is slower: keep
the number of stars small. The catalog records the aberrations used. Left: the extra
Zernikes of each star; right: the size of each star's truth PSF.

In [ ]:
s = RomanSimulation(n_stars=6, seed=11)
s.set_placement("grid")
s.add_aberration_scatter(0.03)
scatter = s.run()
cat = scatter.catalogs[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for k in (4, 5, 6, 7, 8, 11):
    axes[0].plot(cat["star_id"], cat[f"z{k}"], "o-", label=f"Z{k}")
axes[0].set(xlabel="star id", ylabel="extra aberration [waves]")
axes[0].legend(ncol=3, fontsize=8)
axes[1].plot(cat["star_id"], [hsm(truth(scatter, int(i)))["T [pix²]"] for i in cat["star_id"]],
             "o")
axes[1].set(xlabel="star id", ylabel="T [pix²] (HSM, oversampled)")
fig.tight_layout()
plt.show()
del scatter

## 7. Variations of the errors

Every error (dilation, shear, MTF scale, kernel, Zernikes) takes two optional variations on
top of its value:

- `sigma_obs`: a Gaussian offset drawn once per observation, shared by all SCAs (a change in
  time);
- `sigma_field`: a random linear field a + b x̂ + c ŷ across each SCA, per observation, with
  x̂ = x / 4088, ŷ = y / 4088 and a, b, c drawn with this σ.

For Zernikes both can be given per Noll index. The catalog has the value used at each star.
`plot_error_field` shows a value across the SCAs; `plot_error_vs_obs` its spread per
observation.

In [ ]:
s = RomanSimulation(scas=[1, 2], n_obs=4, n_stars=25, seed=12)
s.set_placement("grid")
s.add_dilation(1.0, sigma_obs=0.005, sigma_field=0.01)
s.set_zernikes({4: 0.0}, sigma_field={5: 0.03})
s.add_kernel(0.02, 0.0, 0.02, sigma_obs=0.005)
varied = s.run()
varied.plot_error_field("dilation", obs=0)
varied.plot_error_field("z5", obs=0)
varied.plot_error_vs_obs("dilation")
varied.plot_error_vs_obs("kernel_mxx");

The resulting truth PSF at every star of observation 0: HSM size and shape across the two
SCAs.

In [ ]:
cat = varied.catalogs[0]
moments = [hsm(truth(varied, int(i))) for i in cat["star_id"]]
del varied
fig, axes = plt.subplots(2, 3, figsize=(13, 7.5))
for j, sca in enumerate([1, 2]):
    sel = cat["sca"] == sca
    for k, q in enumerate(("T [pix²]", "e1", "e2")):
        v = np.array([m[q] for m in moments])
        sc = axes[j, k].scatter(cat["x"][sel], cat["y"][sel], c=v[sel], s=100,
                                vmin=v.min(), vmax=v.max())
        axes[j, k].set(title=f"SCA {sca}: {q}", aspect="equal", xlim=(0, 4089),
                       ylim=(0, 4089))
        fig.colorbar(sc, ax=axes[j, k])
fig.tight_layout()
plt.show()

## 8. Several SCAs, several observations, and the WCS

`scas=[...]` simulates several detectors; `n_obs` repeats the exposure. By default the
observations share the set-up and differ only through their random draws (positions,
magnitudes, noise, and any `sigma_obs` offsets). `sim.catalog(obs, sca)` gives the stars of
one image.

Two WCS options:
- `set_wcs("simple")` (default): a plain 0.11″ pixel scale; every SCA has the same sky
  coordinates `u`, `v` (arcsec).
- `set_wcs("roman", ra, dec, pa, date)`: the Roman WCS from `galsim.roman.getWCS`, with each
  SCA at its place in the focal plane, including distortion; catalogs have `ra`, `dec`.

In [ ]:
s = RomanSimulation(scas=[1, 10], n_obs=2, n_stars=15, seed=4)
s.set_brightness(mag_range=(17, 20))
multi = s.run()
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for i in range(2):
    for j, sca in enumerate([1, 10]):
        multi.plot_sca(obs=i, sca=sca, ax=axes[i, j])
fig.tight_layout()
plt.show()
display(multi.catalog(obs=1, sca=10)["star_id", "obs", "sca", "x", "y", "mag"][:5])
del multi

With the Roman WCS: the 18 SCAs on the sky around the pointing (tangent plane), and a
simulation on one SCA with `ra`, `dec` in its catalog.

In [ ]:
from psf_fomo.sim.wcs import make_wcs

builder = RomanSimulation(scas=list(range(1, 19)))
builder.set_wcs("roman", ra=61.24, dec=-48.42)
wcs, pointing = make_wcs(builder.config["wcs"], builder.config["scas"])
fig, ax = plt.subplots(figsize=(6.5, 5.5))
corners = [galsim.PositionD(1, 1), galsim.PositionD(4088, 1), galsim.PositionD(4088, 4088),
           galsim.PositionD(1, 4088), galsim.PositionD(1, 1)]
for sca, w in wcs.items():
    uv = [pointing.project(w.toWorld(c)) for c in corners]
    u = [p[0] / galsim.degrees for p in uv]
    v = [p[1] / galsim.degrees for p in uv]
    ax.plot(u, v, "k-")
    ax.text(np.mean(u[:4]), np.mean(v[:4]), str(sca), ha="center", va="center")
ax.set(aspect="equal", xlabel="u [deg]", ylabel="v [deg]", title="Roman SCAs on the sky")
ax.invert_xaxis()
plt.show()

s = RomanSimulation(scas=[9], n_stars=9, seed=17)
s.set_placement("grid")
s.set_wcs("roman")
roman = s.run()
display(roman.catalogs[0]["star_id", "x", "y", "ra", "dec"][:4])
del roman

## 9. Colour: the SED of the stars

The optical PSF depends on wavelength, and by default every star is drawn through the
bandpass weighted by its spectral energy distribution (SED). One SED is shared by all stars:

- `set_chromatic("flat")` (default): constant photons per unit wavelength;
- `set_chromatic("my_sed.dat", wave_type="nm", flux_type="flambda")`: an SED file;
- `set_chromatic(galsim.SED(...))`: any GalSim SED (stored as a table over the filter);
- `set_chromatic(None)`: monochromatic, the PSF at the filter's effective wavelength
  (faster).

Below: one star with a redder SED (photons rising with wavelength) and drawn
monochromatically, each compared with the default flat SED: (PSF − flat-SED PSF) / peak.

In [ ]:
red_sed = galsim.SED(galsim.LookupTable([500.0, 2500.0], [0.2, 2.0], interpolant="linear"),
                     wave_type="nm", flux_type="fphotons")
flat = one_star_truth()
colour_psfs = {"red SED": one_star_truth(lambda s: s.set_chromatic(red_sed)),
               "monochromatic": one_star_truth(lambda s: s.set_chromatic(None))}
show_differences(flat, list(colour_psfs.values()), list(colour_psfs))
Table(rows=[{"PSF": "flat SED (default)", **hsm(flat)}]
      + [{"PSF": name, **hsm(im)} for name, im in colour_psfs.items()])

## 10. Stars for Piff

`sim.stars(sca, obs)` cuts `stamp_size` stamps from the images and returns Piff `Star`
objects (weight 1 / background variance; properties `sca`, `chipnum`, `star_id`, `obs`) and
the matching catalog rows. This is what the moments and Piff validation modules use.

In [ ]:
stars, rows = bright.stars(sca=1)
star = stars[0]
print(len(stars), "stars; stamp", star.image.array.shape)
print("properties:", {k: star[k] for k in ("sca", "chipnum", "star_id", "obs")})
print("weight = 1 / bkg_var:", star.weight.array[0, 0], 1 / rows["bkg_var"][0])
bright.plot_stamps(n_max=8);

## 11. Saving and configurations

The whole configuration is a plain dict, `builder.config`, which can be written to YAML and
read back; the same configuration (including the seed) gives the same images.

The cells below show how to save; they are commented out so that running the notebook
writes nothing. A saved simulation has one FITS image file per observation (one HDU per
SCA), one catalog per observation, and `config.yaml` (configuration and software versions).

In [ ]:
builder = RomanSimulation(scas=[3], n_stars=5, seed=18)
builder.add_dilation(1.01)
builder.set_noise("roman")
print(yaml.safe_dump(to_plain(builder.config), sort_keys=False))

In [ ]:
# Save a configuration, and rebuild the simulation from it:
# builder.to_yaml("my_simulation.yaml")
# builder = RomanSimulation.from_yaml("my_simulation.yaml")

# Save the images, catalogs and configuration while running, or afterwards:
# sim = builder.run(save_to="out/my_simulation", save_noiseless=True)
# sim.save("out/my_simulation")

# Reload a saved simulation (the same `Simulation` object as `run` returns):
# from psf_fomo.sim import load_simulation
# sim = load_simulation("out/my_simulation")

Rebuilding from the configuration dict reproduces the images exactly (here without writing a
file):

In [ ]:
a = builder.run()
b = RomanSimulation.from_config(builder.config).run()
print("identical images:", np.array_equal(a.images[0, 3].array, b.images[0, 3].array))
del a, b

## 12. Everything together

Two SCAs, three observations, the Roman WCS and detector effects, a magnitude range, an
aberration field across each SCA that changes between observations, a dilation that changes
between observations, a small Gaussian kernel and jitter. The plots show one image of each
SCA, some stamps, an aberration across the SCAs and over the observations, and the truth PSF
size and shape of every star.

In [ ]:
s = RomanSimulation(scas=[1, 7], n_obs=3, n_stars=40, seed=19)
s.set_brightness(mag_range=(17, 21))
s.set_wcs("roman")
s.set_noise("roman")
s.set_zernikes({4: 0.02}, sigma_obs=0.005, sigma_field={5: 0.02, 6: 0.02})
s.add_dilation(1.0, sigma_obs=0.004)
s.add_kernel(0.01, 0.0, 0.01)
s.add_jitter("random_walk")
full = s.run()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
full.plot_sca(obs=0, sca=1, ax=axes[0])
full.plot_sca(obs=0, sca=7, ax=axes[1])
plt.show()
full.plot_stamps(obs=0, sca=7, n_max=8)
full.plot_error_field("z5", obs=0)
full.plot_error_vs_obs("z4");

In [ ]:
cat = full.catalog(obs=0)
moments = [hsm(truth(full, int(i))) for i in cat["star_id"]]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, q in zip(axes, ("T [pix²]", "e1", "e2")):
    v = np.array([m[q] for m in moments])
    for sca, marker in ((1, "o"), (7, "s")):
        sel = cat["sca"] == sca
        ax.scatter(cat["mag"][sel], v[sel], marker=marker, s=15, label=f"SCA {sca}")
    ax.set(xlabel="AB magnitude", ylabel=q, title=f"truth PSF {q}")
    ax.legend()
fig.tight_layout()
plt.show()

## Next

- `02_moments.ipynb`: measuring PSF size and shape on these images.
- `03_piff_validation.ipynb`: fitting them with Piff and checking the result.